# Previsão diária do volume de tráfego com SARIMAX

Este notebook lê `VolumeDiario.csv`, agrega motocicletas, veículos de passeio e veículos comerciais por dia, valida o modelo nos 30 dias mais recentes e gera a previsão futura com intervalo de confiança de 95%.

O modelo usa o logaritmo do volume para reduzir o efeito de valores extremos, sazonalidade semanal e variáveis externas para sazonalidade anual e quantidade de pontos de medição.

In [ ]:
from pathlib import Path

_requirements = next(
    (path for path in [Path("../requirements.txt"), Path("RoadFixerAPI/requirements.txt")] if path.exists()),
    None,
)
if _requirements is None:
    raise FileNotFoundError("requirements.txt não encontrado.")
_requirements = str(_requirements)
%pip install -r $_requirements

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

HORIZONTE_PREVISAO = 30
HORIZONTE_VALIDACAO = 30
JANELA_TREINO_DIAS = 730
ORDEM = (1, 1, 1)
ORDEM_SAZONAL = (0, 1, 1, 7)
LIMIAR_DIA_INCOMPLETO = 0.60
COLUNAS_VOLUME = ["QTD_MOTO", "QTD_PASSEIO", "QTD_COMERCIAL"]

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Carregamento e validação do CSV

In [ ]:
def localizar_volume_diario() -> Path:
    cwd = Path.cwd().resolve()
    candidatos_relativos = [
        Path("Database/In/volumediario.csv"),
        Path("Database/In/VolumeDiario.csv"),
        Path("Database/Processed/volumediario.csv"),
        Path("Database/Processed/VolumeDiario.csv"),
        Path("RoadFixerAPI/Database/In/volumediario.csv"),
        Path("RoadFixerAPI/Database/In/VolumeDiario.csv"),
        Path("RoadFixerAPI/Database/Processed/volumediario.csv"),
        Path("RoadFixerAPI/Database/Processed/VolumeDiario.csv"),
    ]
    bases = [cwd, cwd.parent]
    candidatos = []
    for base in bases:
        for relativo in candidatos_relativos:
            candidato = (base / relativo).resolve()
            if candidato not in candidatos:
                candidatos.append(candidato)

    for candidato in candidatos:
        if candidato.is_file():
            return candidato

    caminhos = "\n".join(f"- {path}" for path in candidatos)
    raise FileNotFoundError(f"VolumeDiario.csv não encontrado. Caminhos verificados:\n{caminhos}")


ARQUIVO_CSV = localizar_volume_diario()
dados_brutos = pd.read_csv(ARQUIVO_CSV, encoding="utf-8-sig")
dados_brutos.columns = dados_brutos.columns.str.strip().str.upper()

colunas_obrigatorias = {"DATA", *COLUNAS_VOLUME}
colunas_ausentes = sorted(colunas_obrigatorias - set(dados_brutos.columns))
if colunas_ausentes:
    raise ValueError(f"Colunas obrigatórias ausentes no CSV: {colunas_ausentes}")

print(f"Arquivo: {ARQUIVO_CSV}")
print(f"Linhas carregadas: {len(dados_brutos):,}")
dados_brutos.head()

In [ ]:
dados = dados_brutos.copy()
dados["DATA"] = pd.to_datetime(dados["DATA"], errors="coerce")
for coluna in COLUNAS_VOLUME:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

linhas_invalidas = dados[["DATA", *COLUNAS_VOLUME]].isna().any(axis=1)
linhas_negativas = (dados[COLUNAS_VOLUME] < 0).any(axis=1)
quantidade_invalidas = int((linhas_invalidas | linhas_negativas).sum())
if quantidade_invalidas:
    warnings.warn(f"{quantidade_invalidas} linhas inválidas foram removidas.")
dados = dados.loc[~(linhas_invalidas | linhas_negativas)].copy()

dados["VOLUME_TOTAL"] = dados[COLUNAS_VOLUME].sum(axis=1)
serie_diaria = (
    dados.groupby("DATA")
    .agg(
        VOLUME_TOTAL=("VOLUME_TOTAL", "sum"),
        PONTOS_MEDICAO=("VOLUME_TOTAL", "size"),
    )
    .sort_index()
)

ultima_data_descartada = None
if len(serie_diaria) >= 29:
    referencia_recente = serie_diaria["VOLUME_TOTAL"].iloc[-29:-1].median()
    proporcao_ultimo_dia = serie_diaria["VOLUME_TOTAL"].iloc[-1] / referencia_recente
    if proporcao_ultimo_dia < LIMIAR_DIA_INCOMPLETO:
        ultima_data_descartada = serie_diaria.index[-1]
        warnings.warn(
            f"O último dia ({ultima_data_descartada.date()}) tem apenas "
            f"{proporcao_ultimo_dia:.1%} do volume mediano recente e foi tratado como incompleto."
        )
        serie_diaria = serie_diaria.iloc[:-1]

indice_completo = pd.date_range(serie_diaria.index.min(), serie_diaria.index.max(), freq="D")
serie_diaria = serie_diaria.reindex(indice_completo)
dias_ausentes = int(serie_diaria["VOLUME_TOTAL"].isna().sum())
if dias_ausentes:
    warnings.warn(f"{dias_ausentes} dias ausentes serão interpolados.")
    serie_diaria["VOLUME_TOTAL"] = serie_diaria["VOLUME_TOTAL"].interpolate(limit_direction="both")
    serie_diaria["PONTOS_MEDICAO"] = serie_diaria["PONTOS_MEDICAO"].interpolate(limit_direction="both")

serie_diaria.index.name = "DATA"
serie_diaria = serie_diaria.asfreq("D")

if len(serie_diaria) < JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO:
    raise ValueError(
        f"São necessários pelo menos {JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO} dias, "
        f"mas o CSV contém {len(serie_diaria)} dias utilizáveis."
    )

print(f"Período utilizável: {serie_diaria.index.min().date()} a {serie_diaria.index.max().date()}")
print(f"Dias utilizáveis: {len(serie_diaria):,}")
print(f"Dias ausentes encontrados: {dias_ausentes}")
serie_diaria.tail()

In [ ]:
ax = serie_diaria["VOLUME_TOTAL"].plot(figsize=(14, 5), color="#2563eb", linewidth=1)
ax.set_title("Volume diário total de tráfego")
ax.set_xlabel("Data")
ax.set_ylabel("Veículos por dia")
plt.tight_layout()
plt.show()

## 2. Funções do modelo

In [ ]:
DATA_REFERENCIA = serie_diaria.index.min()


def criar_variaveis_externas(indice, pontos_medicao, pontos_referencia):
    indice = pd.DatetimeIndex(indice)
    dias = (indice - DATA_REFERENCIA).days.to_numpy()
    pontos = np.asarray(pontos_medicao, dtype=float)
    return pd.DataFrame(
        {
            "PONTOS_MEDICAO_REL": pontos / float(pontos_referencia) - 1.0,
            "ANO_SEN": np.sin(2 * np.pi * dias / 365.25),
            "ANO_COS": np.cos(2 * np.pi * dias / 365.25),
        },
        index=indice,
    )


def ajustar_sarimax(serie, variaveis_externas):
    modelo = SARIMAX(
        np.log1p(serie.astype(float)),
        exog=variaveis_externas,
        order=ORDEM,
        seasonal_order=ORDEM_SAZONAL,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return modelo.fit(disp=False, maxiter=200)


def voltar_escala_original(valores):
    return np.clip(np.expm1(valores), a_min=0, a_max=None)

## 3. Validação temporal nos últimos 30 dias

In [ ]:
dados_validacao = serie_diaria.tail(JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO)
treino = dados_validacao.iloc[:-HORIZONTE_VALIDACAO]
teste = dados_validacao.iloc[-HORIZONTE_VALIDACAO:]
pontos_referencia_validacao = treino["PONTOS_MEDICAO"].median()

exog_treino = criar_variaveis_externas(
    treino.index, treino["PONTOS_MEDICAO"], pontos_referencia_validacao
)
exog_teste = criar_variaveis_externas(
    teste.index, teste["PONTOS_MEDICAO"], pontos_referencia_validacao
)

modelo_validacao = ajustar_sarimax(treino["VOLUME_TOTAL"], exog_treino)
previsao_validacao_log = modelo_validacao.get_forecast(
    steps=HORIZONTE_VALIDACAO, exog=exog_teste
).predicted_mean
previsao_validacao = pd.Series(
    voltar_escala_original(previsao_validacao_log.to_numpy()),
    index=teste.index,
    name="PREVISAO_SARIMAX",
)

observado = teste["VOLUME_TOTAL"]
mae = mean_absolute_error(observado, previsao_validacao)
rmse = np.sqrt(mean_squared_error(observado, previsao_validacao))
mape = (np.abs((observado - previsao_validacao) / observado).mean() * 100)
baseline_semanal = dados_validacao["VOLUME_TOTAL"].shift(7).loc[teste.index]
mae_baseline = mean_absolute_error(observado, baseline_semanal)

metricas = pd.Series(
    {
        "MAE_SARIMAX": mae,
        "RMSE_SARIMAX": rmse,
        "MAPE_SARIMAX_PERCENTUAL": mape,
        "MAE_BASELINE_7_DIAS": mae_baseline,
        "GANHO_MAE_PERCENTUAL": (1 - mae / mae_baseline) * 100,
    }
)
display(metricas.to_frame("VALOR").round(2))

comparacao_validacao = pd.DataFrame(
    {
        "OBSERVADO": observado,
        "PREVISAO_SARIMAX": previsao_validacao,
        "BASELINE_7_DIAS": baseline_semanal,
    }
)
comparacao_validacao.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(comparacao_validacao.index, comparacao_validacao["OBSERVADO"], label="Observado", color="#111827", linewidth=2)
ax.plot(comparacao_validacao.index, comparacao_validacao["PREVISAO_SARIMAX"], label="Previsão SARIMAX", color="#2563eb", linewidth=2)
ax.plot(comparacao_validacao.index, comparacao_validacao["BASELINE_7_DIAS"], label="Baseline 7 dias", color="#9ca3af", linewidth=1)
ax.set_title("Validação temporal do SARIMAX")
ax.set_xlabel("Data")
ax.set_ylabel("Veículos por dia")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Treino final e previsão futura

In [ ]:
treino_final = serie_diaria.tail(JANELA_TREINO_DIAS)
pontos_referencia_final = treino_final["PONTOS_MEDICAO"].median()
exog_final = criar_variaveis_externas(
    treino_final.index, treino_final["PONTOS_MEDICAO"], pontos_referencia_final
)
modelo_final = ajustar_sarimax(treino_final["VOLUME_TOTAL"], exog_final)

indice_futuro = pd.date_range(
    treino_final.index.max() + pd.Timedelta(days=1),
    periods=HORIZONTE_PREVISAO,
    freq="D",
)
pontos_futuros = int(round(treino_final["PONTOS_MEDICAO"].tail(28).median()))
exog_futuro = criar_variaveis_externas(
    indice_futuro,
    np.repeat(pontos_futuros, HORIZONTE_PREVISAO),
    pontos_referencia_final,
)

resultado_futuro = modelo_final.get_forecast(
    steps=HORIZONTE_PREVISAO, exog=exog_futuro
)
previsao_futura = voltar_escala_original(resultado_futuro.predicted_mean.to_numpy())
intervalo_log = resultado_futuro.conf_int(alpha=0.05)
limite_inferior = voltar_escala_original(intervalo_log.iloc[:, 0].to_numpy())
limite_superior = voltar_escala_original(intervalo_log.iloc[:, 1].to_numpy())

previsao = pd.DataFrame(
    {
        "PREVISAO_VOLUME": np.rint(previsao_futura).astype(np.int64),
        "LIMITE_INFERIOR_95": np.rint(limite_inferior).astype(np.int64),
        "LIMITE_SUPERIOR_95": np.rint(limite_superior).astype(np.int64),
        "PONTOS_MEDICAO_ASSUMIDOS": pontos_futuros,
    },
    index=indice_futuro,
)
previsao.index.name = "DATA"
previsao

In [ ]:
historico_grafico = serie_diaria["VOLUME_TOTAL"].tail(90)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(historico_grafico.index, historico_grafico, label="Histórico", color="#111827")
ax.plot(previsao.index, previsao["PREVISAO_VOLUME"], label="Previsão SARIMAX", color="#2563eb")
ax.fill_between(
    previsao.index,
    previsao["LIMITE_INFERIOR_95"],
    previsao["LIMITE_SUPERIOR_95"],
    color="#93c5fd",
    alpha=0.35,
    label="Intervalo de 95%",
)
ax.axvline(serie_diaria.index.max(), color="#6b7280", linestyle="--", linewidth=1)
ax.set_title(f"Previsão do volume de tráfego para os próximos {HORIZONTE_PREVISAO} dias")
ax.set_xlabel("Data")
ax.set_ylabel("Veículos por dia")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Exportação do modelo e das previsões

In [ ]:
PASTA_ARTEFATOS = ARQUIVO_CSV.parents[2] / "ModeloEstatistico" / "artifacts"
PASTA_ARTEFATOS.mkdir(parents=True, exist_ok=True)

caminho_modelo = PASTA_ARTEFATOS / "sarimax_volume_trafego.joblib"
caminho_previsao = PASTA_ARTEFATOS / "previsao_volume_trafego.csv"

pacote_modelo = {
    "modelo": modelo_final,
    "ordem": ORDEM,
    "ordem_sazonal": ORDEM_SAZONAL,
    "colunas_volume": COLUNAS_VOLUME,
    "pontos_referencia": float(pontos_referencia_final),
    "pontos_futuros_assumidos": pontos_futuros,
    "data_referencia": DATA_REFERENCIA,
    "ultima_data_treino": treino_final.index.max(),
    "metricas_validacao": metricas.to_dict(),
}
joblib.dump(pacote_modelo, caminho_modelo, compress=3)
previsao.to_csv(caminho_previsao, encoding="utf-8-sig")

print(f"Modelo salvo em: {caminho_modelo.resolve()}")
print(f"Previsões salvas em: {caminho_previsao.resolve()}")